# Task 14 — symWeighted independent recovery clone

This fail-closed recovery runner creates one new clone and runs normal and optimized Python against the immutable SHA. It compares the resulting semantic output hashes to the already completed independent clone-a witness.

In [ ]:
import datetime, hashlib, json, os, platform, shlex, subprocess, tempfile, time, urllib.request
from pathlib import Path

REPO_URL = 'https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git'
TARGET_SHA = '2c009f607a6e0747f69effc0378b577b0f853ffb'
JUDGE_REL = 'scripts/judge_spatial_symweighted_factorization.py'
HARNESS_REL = 'scripts/run_spatial_symweighted_gate_detached.ps1'
JUDGE_SHA256 = 'a95e66da0ee527b1776ceb3d13d83760d1fd88cc9227ebea668a2b98ca1946cf'
HARNESS_SHA256 = '668681a7f29e2a228b8036be31472a0203d56edfd7f9de9a8753dc0449bc5a65'
EXPECTED_COUNT = 5460
EXPECTED_HASHES = {
    'log': 'a5bc995539a70b8e071cf71bd568d956248e978a1647621c96e4d415f2e322ba',
    'stderr': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855',
    'exitcode': '9a271f2a916b0b6ee6cecb2426f0b3206ef074578be55d9bc94f6f3fe3ab86aa',
}

def fail(message, payload=None):
    print(f'RESULT: FAIL: {message}')
    if payload is not None:
        print(payload)
    raise RuntimeError(message)

def sha256_bytes(data): return hashlib.sha256(data).hexdigest()
def sha256_file(path): return sha256_bytes(path.read_bytes())

def run_checked(command):
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
    print('$', shlex.join(map(str, command)))
    print(result.stdout, end='')
    print(f'[exit {result.returncode}]')
    if result.returncode != 0: fail('setup command failed', {'command': command, 'exit': result.returncode})
    return result.stdout.strip()

remote_hashes = {}
for rel, expected in ((JUDGE_REL, JUDGE_SHA256), (HARNESS_REL, HARNESS_SHA256)):
    url = f'https://raw.githubusercontent.com/lluiseriksson/THE-ERIKSSON-PROGRAMME/{TARGET_SHA}/{rel}'
    actual = sha256_bytes(urllib.request.urlopen(url, timeout=60).read())
    print(f'remote {TARGET_SHA} {rel} sha256={actual}')
    if actual != expected: fail('remote source hash mismatch', {'path': rel, 'actual': actual})
    remote_hashes[rel] = actual

root = Path(tempfile.mkdtemp(prefix='spatial-symweighted-recovery-clone-'))
repo = root / 'repo'
outputs = root / 'outputs'
outputs.mkdir()
run_checked(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(repo)])
run_checked(['git', '-C', str(repo), 'checkout', '--detach', TARGET_SHA])
head = run_checked(['git', '-C', str(repo), 'rev-parse', 'HEAD'])
if head != TARGET_SHA: fail('clone HEAD mismatch', {'head': head})
if sha256_file(repo / JUDGE_REL) != JUDGE_SHA256 or sha256_file(repo / HARNESS_REL) != HARNESS_SHA256:
    fail('clone source hash mismatch')

record = {
    'status': None, 'classification': 'exact symWeighted recovery clone',
    'target_sha': TARGET_SHA, 'remote_hashes': remote_hashes,
    'runtime': platform.platform(), 'python': platform.python_version(), 'cpu_count': os.cpu_count(),
    'utc_started': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'root': str(root), 'repo': str(repo), 'head': head, 'modes': [],
    'expected_independent_clone_hashes': EXPECTED_HASHES,
}

for mode in ('normal', 'optimized'):
    log_path = outputs / f'{mode}.log'
    stderr_path = outputs / f'{mode}.stderr.log'
    exit_tmp = outputs / f'{mode}.exitcode.tmp'
    exit_path = outputs / f'{mode}.exitcode'
    pids_path = outputs / f'{mode}.pids.json'
    runner_path = outputs / f'{mode}.runner.sh'
    if any(p.exists() for p in (log_path, stderr_path, exit_tmp, exit_path, pids_path, runner_path)):
        fail('stale mode output exists before launch', {'mode': mode})
    flags = '-O ' if mode == 'optimized' else ''
    runner = f'''#!/usr/bin/env bash
set -u
printf '{{"wrapper_pid":%s' "$$" > {shlex.quote(str(pids_path))}
python {flags}{shlex.quote(str(repo / JUDGE_REL))} > {shlex.quote(str(log_path))} 2> {shlex.quote(str(stderr_path))} &
child=$!
printf ',"child_pid":%s}}\n' "$child" >> {shlex.quote(str(pids_path))}
wait "$child"
code=$?
printf '%s\n' "$code" > {shlex.quote(str(exit_tmp))}
test -s {shlex.quote(str(exit_tmp))}
grep -Eq '^-?[0-9]+$' {shlex.quote(str(exit_tmp))}
mv {shlex.quote(str(exit_tmp))} {shlex.quote(str(exit_path))}
exit "$code"
'''
    runner_path.write_text(runner, encoding='utf-8', newline='\n')
    started = datetime.datetime.now(datetime.timezone.utc).isoformat()
    t0 = time.perf_counter()
    process = subprocess.Popen(['bash', str(runner_path)], cwd=repo)
    shell_exit = process.wait()
    wall_seconds = time.perf_counter() - t0
    completed = datetime.datetime.now(datetime.timezone.utc).isoformat()
    if not exit_path.exists() or exit_path.stat().st_size == 0: fail('missing or empty exit code', {'mode': mode})
    exit_lines = exit_path.read_text(encoding='utf-8').splitlines()
    if len(exit_lines) != 1: fail('exit code is not exactly one line', {'mode': mode})
    try: persisted_exit = int(exit_lines[0])
    except ValueError: fail('exit code is not decimal', {'mode': mode})
    if shell_exit != 0 or persisted_exit != shell_exit:
        fail('nonzero or discordant exit', {'mode': mode, 'shell': shell_exit, 'persisted': persisted_exit})
    lines = log_path.read_text(encoding='utf-8').splitlines()
    if len(lines) != 1: fail('stale or extra log output', {'mode': mode, 'lines': len(lines)})
    payload = json.loads(lines[0])
    fields = ('configuration_pairs_checked', 'scale_mutations_rejected',
              'source_closing_bond_mutations_rejected', 'target_closing_bond_mutations_rejected')
    if payload.get('status') != 'PASS' or any(payload.get(k) != EXPECTED_COUNT for k in fields):
        fail('PASS payload or counters invalid', {'mode': mode, 'payload': payload})
    if payload.get('ring_sizes') != [1,2,3,4,5,6] or stderr_path.read_bytes() != b'':
        fail('ring sizes or stderr invalid', {'mode': mode})
    pids = json.loads(pids_path.read_text(encoding='utf-8'))
    if pids.get('wrapper_pid') != process.pid or not isinstance(pids.get('child_pid'), int):
        fail('PID record mismatch', {'mode': mode, 'pids': pids})
    hashes = {'log': sha256_file(log_path), 'stderr': sha256_file(stderr_path), 'exitcode': sha256_file(exit_path)}
    if hashes != EXPECTED_HASHES: fail('independent clone hash mismatch', {'mode': mode, 'hashes': hashes})
    mode_record = {'mode': mode, 'started': started, 'completed': completed,
        'wall_seconds': wall_seconds, 'wrapper_pid': process.pid, 'child_pid': pids['child_pid'],
        'shell_exit': shell_exit, 'persisted_exit': persisted_exit,
        'fresh_outputs_verified': True, 'hashes': hashes, 'payload': payload}
    record['modes'].append(mode_record)
    print(json.dumps(mode_record, sort_keys=True))

if record['modes'][0]['hashes'] != record['modes'][1]['hashes']:
    fail('normal/optimized recovery hashes disagree')
record['status'] = 'PASS'
record['utc_completed'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
artifact = Path('/content/spatial_symweighted_recovery_clone_certificate.json')
artifact.write_text(json.dumps(record, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(f'certificate_sha256={sha256_file(artifact)}')
print('SPATIAL SYMWEIGHTED RECOVERY CLONE PASS')
from google.colab import files
files.download(str(artifact))